In [1]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

/media/aumoza/Strg_1/Freelance/personal/Testing_Embeddings_For_recommendation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pc = Pinecone(api_key=os.getenv("pinecone_api_key"))

In [3]:
pc.create_index(
  name="social-media",
  dimension=768,
  metric="cosine",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  )
)

ForbiddenException: (403)
Reason: Forbidden
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-10', 'x-cloud-trace-context': '8198fe5ac3696ccc04ec430c32883a6f', 'date': 'Wed, 29 Apr 2026 05:34:01 GMT', 'server': 'Google Frontend', 'Content-Length': '257', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"FORBIDDEN","message":"Request failed. You've reached the max serverless indexes allowed in project Default (5). Use namespaces to partition your data into logical groups, or upgrade your plan to add more serverless indexes."},"status":403}


In [4]:
index_name = "social-media"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# 3. Generate and Insert Embeddings
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
text_data = ["What is quantum computing?", "How does machine learning work?"]
embeddings = model.encode(text_data).tolist() # Convert to list for Pinecone

# # Upsert with IDs and optional metadata
# vectors = [
#     {"id": f"vec{i}", "values": emb, "metadata": {"text": text}} 
#     for i, (emb, text) in enumerate(zip(embeddings, text_data))
# ]
# index.upsert(vectors=vectors)

/media/aumoza/Strg_1/Freelance/personal/Testing_Embeddings_For_recommendation/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3552.89it/s]


In [12]:
vectors_to_upsert = [
    {
        "id": "item_1005",  # Your specific custom ID
        "values": embeddings[0], 
        "metadata": {
            "text": "what is quantum computing?"
        }
    },
    {
        "id": "item_1006",
        "values": embeddings[1],
        "metadata": {
            "text": "How does machine learning work?"
        }
    }
]

index.upsert(vectors=vectors_to_upsert)

UpsertResponse(upserted_count=2, _response_info={'raw_headers': {'date': 'Wed, 29 Apr 2026 04:48:17 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '6', 'x-pinecone-request-logical-size': '3171', 'x-pinecone-request-latency-ms': '395', 'x-envoy-upstream-service-time': '166', 'x-pinecone-response-duration-ms': '399', 'grpc-status': '0', 'server': 'envoy'}})

## Retrieve

In [8]:
query_text = "Learning backend right now"
query_vector = model.encode(query_text).tolist()

In [9]:
# 2. Query Pinecone for the top matches
results = index.query(
    vector=query_vector,
    top_k=3,                  # Number of related results to get
    include_values=True,      # Returns the raw embedding values
    include_metadata=True     # Returns your custom ID and stored text
)

In [10]:
for match in results['matches']:
    print(f"ID: {match['id']}")
    print(f"Score (Similarity): {match['score']}")
    print(f"Stored Text: {match['metadata']['text']}")

ID: item_1001
Score (Similarity): 0.583320677
Stored Text: Preparing pizza tonight!
ID: vec0
Score (Similarity): 0.583320677
Stored Text: This is a test document
ID: item_1003
Score (Similarity): 0.583320677
Stored Text: Life update
